# Econometrics Lab 5 — Endogeneity and Instrumental Variables

**Sorbonne Data Analytics**

**Question of the day:** what is the effect of one more year of education on wages?

The problem: people with more *ability* study longer **and** earn more. Ability is not in the data, so it sits in the error term, and education is **endogenous**: $E(X'u) \neq 0$.

| Exercise | Slides |
|---|---|
| 1. Why OLS fails: omitted variable and measurement error | 10–13 |
| 2. First stage: are the instruments relevant? | 18, 26–28, 40 |
| 3. The IV estimator by hand | 22, 30 |
| 4. 2SLS with two OLS regressions, and its trap | 33–34 |
| 5. Hausman and Sargan tests | 35–39 |
| 6. Weak instruments | 40 |

**Python reminder**

| Code | Meaning |
|---|---|
| `A.T`, `A @ B`, `np.linalg.inv(A)` | $A'$, $AB$, $A^{-1}$ |
| `sm.add_constant(df[["x1", "x2"]])` | the matrix $X$ = (constant, x1, x2) |
| `sm.OLS(y, X).fit()` | OLS of $y$ on $X$ |
| `.params`, `.bse`, `.resid`, `.fittedvalues` | coefficients, standard errors, residuals, predictions |

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.sandbox.regression.gmm import IV2SLS

rng = np.random.default_rng(42)

## The data (simulated, so we know the truth)

$$\log(wage) = 1 + \underbrace{0.08}_{b_{educ}}\, educ + 0.02\, exper + \underbrace{0.15\, ability + \varepsilon}_{u\ \text{(error term)}}$$

$$educ = 12 + 1.0\, ability - 0.3\, dist + 0.5\, q4 + v$$

- `ability` is **not observed** by the econometrician: it is in $u$ **and** it drives `educ` → endogeneity.
- `dist` (distance to the nearest university, in 10 km) and `q4` (born in the 4th quarter, as in Angrist–Krueger) change education but have **no direct effect on wages** and are unrelated to ability: these are our **instruments**.
- `exper` (years of experience) is exogenous.

In [ ]:
def simulate_wages(N):
    ability = rng.normal(0, 1, N)  # unobserved
    exper = rng.uniform(0, 20, N)
    dist = rng.uniform(0, 10, N)
    q4 = rng.binomial(1, 0.25, N)
    educ = 12 + 1.0 * ability - 0.3 * dist + 0.5 * q4 + rng.normal(0, 1.5, N)
    lwage = 1 + 0.08 * educ + 0.02 * exper + 0.15 * ability + rng.normal(0, 0.3, N)
    return pd.DataFrame(
        {
            "lwage": lwage,
            "educ": educ,
            "exper": exper,
            "dist": dist,
            "q4": q4,
            "ability": ability,
        }
    )


df = simulate_wages(2000)
df.head()

---
## Exercise 1 — Why OLS fails (slides 10–13)

**(a) Omitted ability.**
1. Regress `lwage` on a constant, `educ` and `exper`. Compare $\hat b_{educ}$ with the true value 0.08.
2. Does the bias disappear when $N$ grows? Re-estimate with $N$ = 100, 1 000, 10 000 and 100 000.
3. Add `ability` to the regression (impossible with real data!). What happens?
4. Check the omitted variable formula: $\text{plim}\ \hat b_{educ} = 0.08 + 0.15\,\dfrac{Cov(educ,\ ability)}{V(educ)}$.

In [ ]:
# 1. OLS without ability
X = sm.add_constant(df[["educ", "exper"]])
ols = ...  # OLS of df["lwage"] on X
print(ols.params)

# 2. Does the bias vanish with more data?
for N in [100, 1_000, 10_000, 100_000]:
    d = simulate_wages(N)
    b_educ = ...  # OLS of d["lwage"] on a constant, educ and exper -> .params["educ"]
    print(f"N = {N:>7}:  b_educ = {b_educ:.4f}")

# 3. OLS with ability
...

# 4. Theoretical probability limit
plim = ...  # hint: np.cov(a, b)[0, 1] is Cov(a, b) and df["educ"].var() is V(educ)
print("Theoretical plim:", plim)

**Your answer:**

**(b) Measurement error.** Forget wages for a moment. The true model is $y = 2x^* + v$ but we only observe $x = x^* + e$, with $V(x^*) = 1$ and $V(e) = 0.25$. Estimate the slope by OLS with $N$ = 100 000 and compare it with the attenuation formula (slide 13):

$$\text{plim}\ \hat b = b\ \frac{V(x^*)}{V(x^*) + V(e)}$$

In [ ]:
N = 100_000
x_star = rng.normal(0, 1, N)  # true variable
x = x_star + rng.normal(0, 0.5, N)  # what we observe: V(e) = 0.25
y = 2 * x_star + rng.normal(0, 1, N)

b_hat = ...  # OLS slope of y on a constant and x (index 1 of .params)
b_theory = ...  # attenuation formula
print(f"OLS slope = {b_hat:.3f}, theory = {b_theory:.3f}, true slope = 2")

**Your answer:**

---
## Exercise 2 — First stage: are the instruments relevant? (slides 18, 26–28, 40)

The instrument matrix $Z$ contains the **excluded instruments** (`dist`, `q4`) **and** the exogenous regressors (constant, `exper`), which are their own instruments. Here $p = 4$ instruments for $k = 3$ regressors.

1. Run the first-stage regression of `educ` on $Z$ = (constant, `exper`, `dist`, `q4`). Are the signs of `dist` and `q4` sensible?
2. Test $H_0: \pi_{dist} = \pi_{q4} = 0$ with an F-test. Rule of thumb: $F < 10$ means weak instruments. (Use the F of the *excluded* instruments only, not the global F of the regression.)
3. The other condition, $E(Z'u) = 0$, cannot be checked with real data. In a simulation we can: compute the correlation between `dist` and `ability`.

In [ ]:
Z = sm.add_constant(df[["exper", "dist", "q4"]])
first = ...  # OLS of df["educ"] on Z
print(first.summary().tables[1])

# 2. F-test on the excluded instruments
print(first.f_test("dist = 0, q4 = 0"))

# 3. Exogeneity of dist (only possible in a simulation!)
...  # hint: np.corrcoef(a, b)[0, 1]

**Your answer:**

---
## Exercise 3 — The IV estimator by hand (slides 22 and 30)

$$\hat b_{IV} = (X'P_Z X)^{-1} X'P_Z\, y, \qquad P_Z = Z(Z'Z)^{-1}Z'$$

$$\hat V(\hat b_{IV}) = \hat\sigma^2_{IV}\,(X'P_Z X)^{-1}, \qquad \hat\sigma^2_{IV} = \frac{SSR_{IV}}{N}, \quad \hat u = y - X\hat b_{IV}$$

1. Complete the function `iv_estimate` (we will reuse it).
2. Compare OLS and IV: which estimate is closer to 0.08? Which one is more precise?
3. Check your results with `IV2SLS(y, X, Z)` from statsmodels.

In [ ]:
def iv_estimate(y, X, Z):
    # Returns the IV coefficients and their asymptotic standard errors
    y, X, Z = np.asarray(y), np.asarray(X), np.asarray(Z)
    Pz = ...  # Z (Z'Z)^(-1) Z'
    b_iv = ...  # (X' Pz X)^(-1) X' Pz y
    u_iv = ...  # y - X b_iv  (with the ORIGINAL X)
    sigma2_iv = ...  # SSR / N
    V_iv = ...  # sigma2_iv (X' Pz X)^(-1)
    return b_iv, np.sqrt(np.diag(V_iv))


b_iv, se_iv = iv_estimate(df["lwage"], X, Z)
print(
    pd.DataFrame(
        {"OLS": ols.params, "IV": b_iv, "se OLS": ols.bse, "se IV": se_iv}
    ).round(4)
)

In [ ]:
# Check with statsmodels
iv = ...  # IV2SLS(y, X, Z).fit()
print(iv.summary())

**Your answer:**

---
## Exercise 4 — 2SLS with two OLS regressions, and its trap (slides 33–34)

2SLS = two OLS regressions:
1. **Stage 1:** regress `educ` on $Z$ and keep the fitted values $\widehat{educ}$ (this is Exercise 2).
2. **Stage 2:** regress `lwage` on a constant, $\widehat{educ}$ and `exper`.

Compare the coefficients **and the standard errors** of stage 2 with those of Exercise 3. Which standard errors are correct, and why?

In [ ]:
df["educ_hat"] = ...  # fitted values of the first stage
stage2 = ...  # OLS of lwage on a constant, educ_hat and exper

print(
    pd.DataFrame(
        {
            "2 OLS": stage2.params.values,
            "IV": b_iv,
            "se 2 OLS": stage2.bse.values,
            "se IV": se_iv,
        },
        index=X.columns,
    ).round(4)
)

**Your answer:**

---
## Exercise 5 — Testing (slides 35–39)

**(a) Is `educ` really endogenous? Hausman test with the augmented regression (slide 37).**
Add the first-stage residuals $\hat v = M_Z\, educ$ to the OLS regression:
$$lwage = b_0 + b_1\, educ + b_2\, exper + c\, \hat v + \varepsilon$$
1. Check that $\hat b_1$ equals $\hat b_{IV}$.
2. Test $H_0: c = 0$ (educ is exogenous) with a t-test. Conclude: should we use OLS or IV?

In [ ]:
df["v_hat"] = ...  # residuals of the first stage
aug = ...  # OLS of lwage on a constant, educ, exper and v_hat
print(aug.summary().tables[1])

**Your answer:**

**(b) Are the instruments valid? Sargan test (slide 39).** With $p$ instruments and $k$ regressors, under $H_0: E(Z'u) = 0$:
$$S = \frac{\hat u' P_Z\, \hat u}{\hat u'\hat u / N} \ \to\ \chi^2_{p-k}, \qquad \hat u = y - X \hat b_{IV}$$
$\hat u' P_Z \hat u$ is the sum of the squared fitted values of the regression of $\hat u$ on $Z$.

1. Complete `sargan_test` and apply it to our instruments ($p - k = 1$ over-identifying restriction).
2. A colleague suggests adding `books` (number of books at home as a child) as an extra instrument. But books are also related to ability… Run the test with `books` added to $Z$. Conclude.

In [ ]:
def sargan_test(y, X, Z):
    b_iv, _ = iv_estimate(y, X, Z)
    u_iv = np.asarray(y) - np.asarray(X) @ b_iv
    fitted = sm.OLS(u_iv, Z).fit().fittedvalues  # this is Pz u
    S = ...  # sum(fitted^2) / (sum(u_iv^2) / N)
    dof = ...  # p - k: number of columns of Z minus number of columns of X
    return S, stats.chi2.sf(S, dof)


S, pval = sargan_test(df["lwage"], X, Z)
print(f"dist and q4:          S = {S:.2f}, p-value = {pval:.3f}")

df["books"] = df["ability"] + rng.normal(0, 1, len(df))  # correlated with ability!
Z_books = sm.add_constant(df[["exper", "dist", "q4", "books"]])
...

**Your answer:**

---
## Exercise 6 — The danger of weak instruments (slide 40)

We simulate 1 000 samples of $N = 500$ with a **single** instrument $z$ of strength $\pi$:
$$educ = 12 + \pi\, z + ability + v$$
For each sample we compute the OLS and IV estimates of $b_{educ}$ (true value 0.08) and the first-stage F-statistic. Compare a **strong** ($\pi = 1$) and a **weak** ($\pi = 0.05$) instrument: mean F, median of the estimates, spread and shape of the IV distribution.

In [ ]:
def one_sample(N, pi):
    ability = rng.normal(0, 1, N)
    z = rng.normal(0, 1, N)
    educ = 12 + pi * z + ability + rng.normal(0, 1.5, N)
    lwage = 1 + 0.08 * educ + 0.15 * ability + rng.normal(0, 0.3, N)

    X = sm.add_constant(educ)
    Z = sm.add_constant(z)
    b_ols = ...  # OLS slope (index 1 of .params)
    b_iv = ...  # IV slope with iv_estimate (index 1 of the coefficients)
    F = ...  # F-statistic of the first stage: .fvalue
    return b_ols, b_iv, F


for pi in [1.0, 0.05]:
    b_ols, b_iv, F = np.array([one_sample(500, pi) for _ in range(1000)]).T
    print(
        f"pi = {pi}: mean F = {F.mean():5.1f} | median OLS = {np.median(b_ols):.3f}"
        f" | median IV = {np.median(b_iv):.3f}"
        f" | 90% of IV in [{np.percentile(b_iv, 5):.2f}, {np.percentile(b_iv, 95):.2f}]"
    )
    plt.hist(b_iv, bins=np.linspace(-0.5, 0.7, 61), alpha=0.5, label=f"IV, pi = {pi}")

plt.axvline(0.08, color="black", linestyle="--", label="true value")
plt.xlabel("IV estimate of b_educ")
plt.legend()
plt.show()

**Your answer:**

---
## Summary

| Question | Tool | Rule |
|---|---|---|
| Is OLS consistent? | Think: omitted variable, measurement error, simultaneity, selection | If $E(X'u) \neq 0$, OLS is biased **and** inconsistent |
| Are the instruments relevant? | First-stage F on the excluded instruments | $F > 10$ |
| Are the instruments exogenous? | Economic argument; Sargan test if $p > k$ | Do not reject $H_0$ |
| Is $X$ really endogenous? | Hausman / augmented regression | Reject → IV; do not reject → OLS (more precise) |
| Standard errors | Proper IV routine (`IV2SLS`) | Never the SE of a manual second stage |